# Qwen3.5 VL Image-to-FOL

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import os
import subprocess
import tempfile
import json
from pathlib import Path
from outlines.types import CFG
from outlines.inputs import Chat
from PIL import Image
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from IPython.display import display, Markdown
import torch
import outlines

In [ ]:
MODEL_ID = "unsloth/Qwen3.5-35B-A3B"
MAX_NEW_TOKENS = 2048

# https://huggingface.co/Qwen/Qwen3.5-0.8B#best-practices
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20
MIN_P = 0.0
PRESENCE_PENALTY = 0.0
REPETITION_PENALTY = 1.1

BASE_DIR = Path.cwd().parent
CATEGORY = "train"

COMPETITION_DATA_DIR = BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

STATE_FILE = BASE_DIR / "smt_state.json"
SMT_FILE = BASE_DIR / "smt.json"

CVC5_PATH = Path.home() / "cvc5-Linux-x86_64-shared" / "bin" / "cvc5"

SUMMARY_CACHE_PATH = BASE_DIR / f"submission_finetuning_summary_{CATEGORY}_state.json"
EXTRACTION_CACHE_PATH = (
    BASE_DIR / f"submission_finetuning_extraction_{CATEGORY}_state.json"
)

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen3.5 (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

In [ ]:
# A stripped down EBNF conforming to strict guided decoding rules.
# 1. Comments are impossible.
# 2. Infinite loops are prevented by a forced exit lifecycle.
# 3. Type-safety is enforced lexically via suffixes/prefixes.
# 4. Hardcoding the preamble signatures
# 5. Prohibit the LLM from re-declaring preamble functions or using generic sorts for Entities/Series
# 6. Use a Phase-Locked Grammar. This forces the LLM to follow a one-way path: first declarations, then data anchors, then logic, then exit.
# 7. We limit underscores and character length to prevent 'series_series_series' loops
# 8. Prevent an infinite token loop by removing the * quantifier and physically cap the number of allowed repetitions.

# ==========================================
# PASS 1A: PURE DECLARATIONS ONLY
# ==========================================
SMT_LIB_GRAMMAR_PASS1A = r"""
?start: script

# 1. Limit the number of declarations to a safe maximum (e.g., 20)
# This prevents the model from rambling until it hits the token limit.
script: decl_line decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line? decl_line?

# 2. Ensure each line is an atomic, unbreakable unit
decl_line: entity_decl | series_decl | real_decl | bool_decl

entity_decl: "(declare-const " ENTITY_SYM " Entity)\n"
series_decl: "(declare-const " SERIES_SYM " Series)\n"
real_decl:   "(declare-const " LOGIC_VAR_REAL " Real)\n"
bool_decl:   "(declare-const " LOGIC_VAR_BOOL " Bool)\n"

ENTITY_SYM: /[a-zA-Z0-9]{1,25}_entity/
SERIES_SYM: /[a-zA-Z0-9]{1,25}_series/
LOGIC_VAR_BOOL: /[a-zA-Z0-9]{1,25}_(drop|inc|dec|const|stable|bool)/
LOGIC_VAR_REAL: /[a-zA-Z0-9]{1,25}_(val|max|min|coord|cycle)/
"""

SMT_CFG_PASS1A = CFG(SMT_LIB_GRAMMAR_PASS1A)

In [ ]:
PROMPT_TEMPLATE_PASS1A = """
<image>

[SUMMARY]
{summary}

[METADATA]
Question Type: {question_type}
Answer Type: {answer_type}
Question: {question}

[TASK]
You are a formal logic extractor (PASS 1A: Declarations). Your goal is to build a "Knowledge Base" using the provided image and summary.
1. Declare all Entities and Series.
2. Pre-declare the logical variables (Bool/Real) you will need to perform reasoning in Pass 2.

[STRICT RULES]
- PURE DECLARATIONS ONLY: Do not perform any logic, and do NOT use `(assert ...)`.
- NO ANCHORS YET: Do NOT extract data points (f series x) or assert names/attributes here. You will do that in the next step.
- SCHEMA LOCK: You must declare any variable (e.g., max_val) here, or you won't be able to use it in Pass 2.
- SERIES: You MUST declare a separate Series variable (e.g., carbon_series, oxygen_series) for every individual data series present in the table. Notice the mandatory '_series' suffix!

[AVAILABLE SMT-LIB ENVIRONMENT]
{preamble}

[EXAMPLE]
{example}
"""

PROMPT_TEMPLATE_PASS1B = """
<image>

[SUMMARY]
{summary}

[TASK]
You are a formal logic extractor (PASS 1B: Assertions & Anchors). Using the variables declared in the Knowledge Base, your task is to:
1. Assert the names and attributes (`name_of`, `attr`) mapping Series to Entities.
2. Extract the exact numeric data points (`f`) for each Series.

[STRICT RULES]
- NO NEW DECLARATIONS: You must only use the exact variables provided below.
- DECIMALS ONLY: SMT-LIB requires explicit decimals for Real types (e.g., 0.0 instead of 0).
- ANCHORS: Extract representative anchors (at least 4) for the declared series.
- UNIQUE VALUES: Each Series can have only ONE value per x-coordinate. Do not assert (f s x y1) and (f s x y2) if y1 != y2.
- CONSISTENCY: Ensure your extracted points form a single, logical curve.

[AVAILABLE SMT-LIB ENVIRONMENT]
{preamble}

[KNOWLEDGE BASE (FROM PASS 1A)]
{declarations}

[EXAMPLE]
{example}
"""

In [ ]:
EXAMPLES_PASS1A = {
    "Yes/No": "(declare-const s2p_entity Entity)\n(declare-const trace_series Series)\n(declare-const max_val Real)\n(declare-const threshold_met_bool Bool)",
    "Factoid": "(declare-const structural_region_entity Entity)\n(declare-const sidewall_entity Entity)\n(declare-const identified_issue_entity Entity)\n(declare-const sidewall_is_part_of_region_bool Bool)\n(declare-const issue_forms_on_sidewall_bool Bool)",
    "List": "(declare-const e_CF2_entity Entity)\n(declare-const e_CF_entity Entity)\n(declare-const e_C_entity Entity)\n(declare-const s_CF2_series Series)\n(declare-const s_CF_series Series)\n(declare-const s_C_series Series)\n(declare-const rank1_entity Entity)\n(declare-const rank2_entity Entity)\n(declare-const rank3_entity Entity)",
    "Paragraph": "(declare-const o1s_entity Entity)\n(declare-const o1s_series Series)\n(declare-const o_initial_drop_bool Bool)\n(declare-const o_steady_decrease_bool Bool)",
}

EXAMPLES_PASS1B = {
    "Yes/No": '(assert (= (name_of s2p_entity) "S2p"))\n(assert (attr trace_series s2p_entity))\n(assert (= (f trace_series 0.0) 0.0))\n(assert (= (f trace_series 10.0) 1.8))\n(assert (= (f trace_series 20.0) 1.6))\n(assert (= (f trace_series 30.0) 1.6))\n(assert (= (f trace_series 50.0) 1.9))\n(assert (= (f trace_series 75.0) 1.2))',
    "Factoid": '(assert (= (name_of structural_region_entity) "Multi-quantum well"))\n(assert (= (name_of sidewall_entity) "sidewall"))\n(assert (= (name_of identified_issue_entity) "Etch damage layer"))',
    "List": '(assert (= (name_of e_CF2_entity) "CF2"))\n(assert (= (name_of e_CF_entity) "CF"))\n(assert (= (name_of e_C_entity) "C"))\n(assert (attr s_CF2_series e_CF2_entity))\n(assert (attr s_CF_series e_CF_entity))\n(assert (attr s_C_series e_C_entity))\n(assert (= (f s_CF2_series 6.25) 5.75))\n(assert (= (f s_CF_series 6.25) 4.15))\n(assert (= (f s_C_series 6.25) 2.50))',
    "Paragraph": '(assert (= (name_of o1s_entity) "O1s"))\n(assert (attr o1s_series o1s_entity))\n(assert (= (f o1s_series 0.0) 16.0))\n(assert (= (f o1s_series 10.0) 10.5))\n(assert (= (f o1s_series 75.0) 7.5))',
}

In [ ]:
PROMPT_TEMPLATE_PASS2 = """
<image>

[SUMMARY]
{summary}

[METADATA]
Question Type: {question_type}
Answer Type: {answer_type}
Question: {question}

[TASK]
You are a formal logic reasoning agent (PASS 2/2: Query). Use ONLY the variables declared in the Knowledge Base to formulate the logical proof for the question based on the provided table and summary.

[STRICT RULES]
- NO DECLARATIONS: Do not use 'declare-const'. Use only the variables provided in the Knowledge Base.
- DIRECT ASSERTIONS: Use the pre-declared Boolean/Real variables to represent your logic.
- FINITE SEARCH: Use (or ...) for maxima/minima iterating ONLY over the exact x-values asserted in the Knowledge Base. Do not hallucinate or guess data points.
- USE PREAMBLE: Use functions like 'is_dec', 'is_inc', and 'is_eq' to assign values to the pre-declared Booleans.
- RANKING/LISTS: If the Answer Type is 'List', you MUST prove the order mathematically by writing individual assertions for the sequence: `(assert (is_gt ...))`. Then bind the entities to `rank1_entity`, `rank2_entity`, etc. DO NOT re-assign the same boolean variables (e.g., `is_dec_bool`) multiple times, as this causes an `unsat` contradiction!
- OUTPUT MATCHES ANSWER TYPE: Ensure your final `(get-value ...)` matches the expected Answer Type format.

[AVAILABLE SMT-LIB ENVIRONMENT]
{preamble}

[KNOWLEDGE BASE (FROM PASS 1)]
{declarations}
{anchors}

[EXAMPLE]
{example}
"""

In [ ]:
EXAMPLES_PASS2 = {
    "Yes/No": "(assert (or (= max_val (f trace_series 0.0)) (= max_val (f trace_series 10.0)) (= max_val (f trace_series 20.0)) (= max_val (f trace_series 30.0)) (= max_val (f trace_series 50.0)) (= max_val (f trace_series 75.0))))\n(assert (and (>= max_val (f trace_series 0.0)) (>= max_val (f trace_series 10.0)) (>= max_val (f trace_series 20.0)) (>= max_val (f trace_series 30.0)) (>= max_val (f trace_series 50.0)) (>= max_val (f trace_series 75.0))))\n(assert (= AnsBool (> max_val 2.0)))\n(check-sat)\n(get-value (AnsBool))\n(exit)",
    "Factoid": '(assert (= sidewall_is_part_of_region_bool true))\n(assert (= issue_forms_on_sidewall_bool true))\n(assert (ite (and sidewall_is_part_of_region_bool issue_forms_on_sidewall_bool) (= AnsString (name_of identified_issue_entity)) (= AnsString "Unknown")))\n(check-sat)\n(get-value (AnsString))\n(exit)',
    "List": "(assert (is_gt s_CF2_series s_CF_series 6.25))\n(assert (is_gt s_CF_series s_C_series 6.25))\n(assert (= rank1_entity e_CF2_entity))\n(assert (= rank2_entity e_CF_entity))\n(assert (= rank3_entity e_C_entity))\n(check-sat)\n(get-value ((name_of rank1_entity) (name_of rank2_entity) (name_of rank3_entity)))\n(exit)",
    "Paragraph": '(assert (= o_initial_drop_bool (is_dec o1s_series 0.0 10.0)))\n(assert (= o_steady_decrease_bool (is_dec o1s_series 10.0 75.0)))\n(assert (= AnsString (ite (and o_initial_drop_bool o_steady_decrease_bool) "Oxygen steadily decreases" "Oxygen fluctuates")))\n(check-sat)\n(get-value (AnsString))\n(exit)',
}

In [ ]:
PREAMBLE = """
;; --- UNIVERSAL PREAMBLE ---
;; Logic: Combined Linear Real Arithmetic and Strings (supported by Z3/CVC5)
(set-logic ALL)

;; 1. TYPE DEFINITIONS
(declare-sort Series 0)
(declare-sort Entity 0)
(declare-const epsilon Real)
(assert (= epsilon 0.001))

;; 2. CORE MAPPING FUNCTIONS

;; Maps a Series and a real input (e.g., time) to a real value
(declare-fun f (Series Real) Real)

;; Indicates whether a Series has a given Entity as an attribute
(declare-fun attr (Series Entity) Bool)

;; Returns the string name of an Entity
(declare-fun name_of (Entity) String)

;; Returns the unit (as string) associated with a Series
(declare-fun unit_of (Series) String)

;; 3. GEOMETRIC & TREND PRIMITIVES

;; Computes absolute difference between two real numbers
(define-fun diff ((a Real) (b Real)) Real
  (ite (>= (- a b) 0.0) (- a b) (- b a)))

;; Checks if Series s1 is significantly greater than s2 at point x (with tolerance epsilon)
(define-fun is_gt ((s1 Series) (s2 Series) (x Real)) Bool
  (> (- (f s1 x) (f s2 x)) epsilon))

;; Checks if Series s1 and s2 are approximately equal at point x (within epsilon)
(define-fun is_eq ((s1 Series) (s2 Series) (x Real)) Bool
  (<= (diff (f s1 x) (f s2 x)) epsilon))

;; Checks if Series s is increasing between x1 and x2 (beyond epsilon threshold)
(define-fun is_inc ((s Series) (x1 Real) (x2 Real)) Bool
  (> (- (f s x2) (f s x1)) epsilon))

;; Checks if Series s is decreasing between x1 and x2 (beyond epsilon threshold)
(define-fun is_dec ((s Series) (x1 Real) (x2 Real)) Bool
  (> (- (f s x1) (f s x2)) epsilon))

;; 4. EXTREMA & INTERSECTION

;; Checks if Series s has value approximately equal to y at point x (within epsilon)
(define-fun is_at_val ((s Series) (x Real) (y Real)) Bool
  (<= (diff (f s x) y) epsilon))

;; Checks if x_m is a local peak (maximum) compared to neighbors x_l and x_r
(define-fun is_peak ((s Series) (x_l Real) (x_m Real) (x_r Real)) Bool
  (and (>= (f s x_m) (f s x_l)) (>= (f s x_m) (f s x_r))))

;; Checks if Series s is approximately constant between x1 and x2
(define-fun is_const ((s Series) (x1 Real) (x2 Real)) Bool
  (<= (diff (f s x1) (f s x2)) epsilon))

;; 5. STANDARDIZED OUTPUT VARIABLES
(declare-const AnsBool Bool)
(declare-const AnsReal Real)
(declare-const AnsString String)
"""

In [ ]:
def validate_smt(code: str) -> tuple[bool, str]:
    """
    Validates SMT-LIB code by executing cvc5.
    Returns: (bool: is_satisfiable, str: output_message)
    """
    # 1. Write to a secure temporary file
    with tempfile.NamedTemporaryFile(mode="w", suffix=".smt2", delete=False) as tf:
        tf.write(code)
        temp_path = tf.name

    try:
        # 2. Execute solver with strict timeout and specific flags
        # --produce-models is necessary if you plan to call (get-model) later
        result = subprocess.run(
            [
                CVC5_PATH,
                "--lang",
                "smt2",
                "--produce-models",
                "--incremental",
                temp_path,
            ],
            capture_output=True,
            text=True,
            timeout=10,
        )

        stdout = result.stdout.strip()
        stderr = result.stderr.strip()

        # 3. Precise Status Parsing
        # The first non-empty line of SMT-LIB output is typically the status
        lines = [line for line in stdout.split("\n") if line.strip()]
        status = lines[0].lower() if lines else ""

        if stderr or "error" in stdout.lower():
            return False, stderr if stderr else stdout
        if "sat" == status:
            return True, stdout
        elif "unsat" == status:
            return False, stdout
        elif "unknown" == status:
            return False, stdout
        else:
            return False, f"Unexpected Solver Output: {stdout}"
    except subprocess.TimeoutExpired:
        return (
            False,
            "Timeout: The solver took too long (potential infinite search space).",
        )
    except Exception as e:
        return False, f"Internal Execution Error: {str(e)}"
    finally:
        # Ensure cleanup even if execution fails
        if os.path.exists(temp_path):
            os.remove(temp_path)

In [ ]:
def clean_duplicate_declarations(declarations_str: str) -> str:
    seen_declarations = set()
    clean_lines = []

    for line in declarations_str.split("\n"):
        match = re.search(
            r"\(declare-const\s+([a-zA-Z0-9_]+)\s+([a-zA-Z0-9_]+)\)", line
        )
        if match:
            var_name = match.group(1)
            var_type = match.group(2)
            signature = f"{var_name}_{var_type}"

            if signature in seen_declarations:
                continue
            seen_declarations.add(signature)

        clean_lines.append(line)

    return "\n".join(clean_lines)


def deduplicate_anchors(anchors_str):
    anchors = {}
    lines = anchors_str.strip().split("\n")
    for line in lines:
        # Regex to find (f series x)
        match = re.search(
            r"\(assert\s+\(=\s+\(f\s+([a-zA-Z0-9_]+)\s+([0-9.]+)\)\s+([0-9.]+)\)\)",
            line,
        )
        if match:
            series, x, y = match.groups()
            anchors[(series, x)] = y

    clean_lines = [f"(assert (= (f {s} {x}) {y}))" for (s, x), y in anchors.items()]
    return "\n".join(clean_lines)


def build_dynamic_phase1b_cfg(declarations: str) -> CFG:
    """
    Pass 1B dynamically forces the model to extract anchors and assert names
    ONLY for the specific variables declared in 1A.
    """
    entities = re.findall(r"\(declare-const\s+([a-zA-Z0-9_]+)\s+Entity\)", declarations)
    series = re.findall(r"\(declare-const\s+([a-zA-Z0-9_]+)\s+Series\)", declarations)
    reals = re.findall(r"\(declare-const\s+([a-zA-Z0-9_]+)\s+Real\)", declarations)

    entity_rule = (
        " | ".join([f'"{e}"' for e in entities]) if entities else '"DUMMY_ENTITY"'
    )
    series_rule = " | ".join([f'"{s}"' for s in series]) if series else '"DUMMY_SERIES"'
    real_rule = " | ".join([f'"{r}"' for r in reals]) if reals else '"DUMMY_REAL"'

    dynamic_grammar = rf"""
    ?start: script
    script: metadata_asserts data_anchors

    metadata_asserts: meta_assert*
    meta_assert: name_assert | attr_assert

    name_assert: "(assert (= (name_of " ENTITY_SYM ") " STRING_LIT "))\n"
    attr_assert: "(assert (attr " SERIES_SYM " " ENTITY_SYM "))\n"

    data_anchors: anchor_assert anchor_assert anchor_assert anchor_assert anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert? anchor_assert?

    anchor_assert: "(assert (= (f " SERIES_SYM " " coordinate_val ") " coordinate_val "))\n"
    coordinate_val: DECIMAL | LOGIC_VAR_REAL

    ENTITY_SYM: {entity_rule}
    SERIES_SYM: {series_rule}
    LOGIC_VAR_REAL: {real_rule}
    DECIMAL: /-?[0-9]+\.[0-9]+/
    STRING_LIT: /"[^"]*"/
    """
    return CFG(dynamic_grammar)


def generate_declarations(
    q_obj, image, summary, max_retries=3, verbose=False, **gen_kwargs
):
    question_text = q_obj.get("question") or q_obj.get("questions")
    question_type = q_obj.get("question_type", "")
    answer_type = q_obj.get("answer_type", "")

    # ==========================================
    # PASS 1A: Generate and Validate Declarations
    # ==========================================
    first_pass_text = PROMPT_TEMPLATE_PASS1A.format(
        question=question_text,
        question_type=question_type,
        answer_type=answer_type,
        summary=summary,
        preamble=PREAMBLE,
        example=EXAMPLES_PASS1A.get(answer_type, ""),
    )

    prompt_pass1a = Chat(
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": first_pass_text},
                ],
            }
        ]
    )

    declarations = ""
    pass1a_success = False

    for attempt in range(max_retries):
        declarations = model(prompt_pass1a, SMT_CFG_PASS1A, **gen_kwargs)
        declarations = declarations.strip()
        declarations = clean_duplicate_declarations(declarations)

        test_smt_pass1a = f""";; --- [PREAMBLE] ---
{PREAMBLE}
;; --- [PASS 1A: Declarations] Attempt {attempt + 1} ---
{declarations}
(check-sat)
"""

        pass1a_success, output = validate_smt(test_smt_pass1a)

        if verbose:
            print(
                f"[PASS 1A - Attempt {attempt + 1}]\n[Code]\n{test_smt_pass1a}\n[Output]\n{output}\n"
            )

        if pass1a_success:
            break

        output_lower = output.lower()
        if "already been defined" in output_lower:
            reflection_text = (
                f"The SMT solver rejected your declarations with this error:\n{output}\n\n"
                f"ERROR: You declared the same variable twice. Remove the duplicate declaration."
            )
        else:
            reflection_text = (
                f"The SMT solver rejected your declarations with this error:\n{output}\n\n"
                f"Please correct the syntax."
            )

        prompt_pass1a.add_assistant_message([{"type": "text", "text": declarations}])
        prompt_pass1a.add_user_message([{"type": "text", "text": reflection_text}])

    if not pass1a_success:
        return None, "Failed to generate valid Pass 1A declarations after retries."

    # ==========================================
    # PASS 1B: Generate and Validate Anchors
    # ==========================================
    try:
        dynamic_cfg_pass1b = build_dynamic_phase1b_cfg(declarations)
    except Exception as e:
        return None, f"Failed to compile dynamic Phase 1B grammar: {e}"

    pass1b_text = PROMPT_TEMPLATE_PASS1B.format(
        summary=summary,
        declarations=declarations,
        example=EXAMPLES_PASS1B.get(answer_type, ""),
    )

    prompt_pass1b = Chat(
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": pass1b_text},
                ],
            }
        ]
    )

    anchors = ""
    pass1b_success = False

    for attempt in range(max_retries):
        anchors = model(prompt_pass1b, dynamic_cfg_pass1b, **gen_kwargs)
        anchors = anchors.strip()
        anchors = deduplicate_anchors(anchors)

        test_smt_pass1b = f""";; --- [PREAMBLE] ---
{PREAMBLE}
;; --- [PASS 1A: Declarations] ---
{declarations}
;; --- [PASS 1B: Anchors] Attempt {attempt + 1} ---
{anchors}
(check-sat)
"""

        pass1b_success, output = validate_smt(test_smt_pass1b)

        if verbose:
            print(
                f"[PASS 1B - Attempt {attempt + 1}]\n[Code]\n{test_smt_pass1b}\n[Output]\n{output}\n"
            )

        if pass1b_success:
            break

        output_lower = output.lower()
        if "unsat" in output_lower:
            reflection_text = (
                "The solver returned 'unsat' (unsatisfiable). Your data mathematically contradicts itself. "
                "Did you assign two different y-values to the same Series at the same x-coordinate? "
            )
        else:
            reflection_text = (
                f"The SMT solver rejected your anchors with this error:\n{output}\n\n"
                f"Please correct the extraction syntax."
            )

        prompt_pass1b.add_assistant_message([{"type": "text", "text": anchors}])
        prompt_pass1b.add_user_message([{"type": "text", "text": reflection_text}])

    if not pass1b_success:
        return None, "Failed to generate valid Pass 1B anchors after retries."

    return declarations, anchors

In [ ]:
def build_dynamic_phase2_cfg(declarations: str) -> CFG:
    entities = re.findall(r"\(declare-const\s+([a-zA-Z0-9_]+)\s+Entity\)", declarations)
    series = re.findall(r"\(declare-const\s+([a-zA-Z0-9_]+)\s+Series\)", declarations)
    bools = re.findall(r"\(declare-const\s+([a-zA-Z0-9_]+)\s+Bool\)", declarations)
    reals = re.findall(r"\(declare-const\s+([a-zA-Z0-9_]+)\s+Real\)", declarations)

    entity_rule = (
        " | ".join([f'"{e}"' for e in entities]) if entities else '"DUMMY_ENTITY"'
    )
    series_rule = " | ".join([f'"{s}"' for s in series]) if series else '"DUMMY_SERIES"'
    bool_rule = " | ".join([f'"{b}"' for b in bools]) if bools else '"DUMMY_BOOL"'
    real_rule = " | ".join([f'"{r}"' for r in reals]) if reals else '"DUMMY_REAL"'

    logic_seq_rule = "logic_assert " + " ".join(["logic_assert?"] * 16)

    dynamic_grammar = rf"""
    ?start: script

    script: logic_sequence final_answer_assert check_sat_cmd get_value_cmd exit_cmd

    # Capped to physically prevent re-assignment loops.
    logic_sequence: {logic_seq_rule}

    logic_assert: "(assert (= " (LOGIC_VAR_BOOL | LOGIC_VAR_REAL) " " term "))\n"
                | "(assert " term_expr ")\n"
                | "(assert " preamble_call ")\n"

    final_answer_assert: "(assert (= " PREAMBLE_CONST " " term "))\n"
    check_sat_cmd: "(check-sat)\n"

    get_value_cmd: "(get-value (" gv_list "))\n"
    gv_list: gv_item (" " gv_item)*
    gv_item: LOGIC_VAR_BOOL | LOGIC_VAR_REAL | PREAMBLE_CONST | ENTITY_SYM | SERIES_SYM | preamble_call
    exit_cmd: "(exit)\n"

    ?term: spec_constant | ENTITY_SYM | SERIES_SYM | LOGIC_VAR_BOOL | LOGIC_VAR_REAL | PREAMBLE_CONST | preamble_call | term_expr
    term_expr: "(" RESERVED_OP " " term_list ")"
    term_list: term (" " term)*
    spec_constant: DECIMAL | NUMERAL | STRING_LIT | "true" | "false"

    preamble_call: "(f " SERIES_SYM " " term ")"
                 | "(attr " SERIES_SYM " " ENTITY_SYM ")"
                 | "(name_of " ENTITY_SYM ")"
                 | "(diff " term " " term ")"
                 | "(is_gt " SERIES_SYM " " SERIES_SYM " " term ")"
                 | "(is_eq " SERIES_SYM " " SERIES_SYM " " term ")"
                 | "(is_inc " SERIES_SYM " " term " " term ")"
                 | "(is_dec " SERIES_SYM " " term " " term ")"
                 | "(is_const " SERIES_SYM " " term " " term ")"
                 | "(is_at_val " SERIES_SYM " " term " " term ")"

    PREAMBLE_CONST: "epsilon" | "AnsBool" | "AnsReal" | "AnsString"
    RESERVED_OP: "=" | "not" | "and" | "or" | ">" | "<" | ">=" | "<=" | "ite"

    ENTITY_SYM: {entity_rule}
    SERIES_SYM: {series_rule}
    LOGIC_VAR_BOOL: {bool_rule}
    LOGIC_VAR_REAL: {real_rule}

    NUMERAL: /-?[0-9]+/
    DECIMAL: /-?[0-9]+\.[0-9]+/
    STRING_LIT: /"[^"]*"/
    """
    return CFG(dynamic_grammar)

In [ ]:
def parse_table_deterministically(table_str: str) -> tuple[str, str]:
    """Parses a Markdown/CSV table and deterministically generates SMT Pass 1A & 1B."""
    lines = [line.strip() for line in table_str.strip().split("\n") if line.strip()]

    # 1. Parse rows (Supports Markdown and fallback to CSV/TSV)
    rows = []
    for line in lines:
        if "|" in line:
            cols = [c.strip() for c in line.split("|")]
            if line.startswith("|"):
                cols = cols[1:]
            if line.endswith("|"):
                cols = cols[:-1]
            if all(c == "" or "-" in c for c in cols):
                continue
            rows.append(cols)

    if not rows:
        for line in lines:
            cols = re.split(r"\t|,", line)
            rows.append([c.strip() for c in cols])

    if not rows:
        return "", ""

    headers = rows[0]
    data = rows[1:]
    y_cols = headers[1:]

    declarations = []
    anchors = []

    # 2. Declare Entities, Series, and Assert Names/Attributes
    for i, y_col in enumerate(y_cols):
        # Sanitize variable identifiers (must be strictly alphanumeric)
        clean_name = re.sub(r"[^a-zA-Z0-9]", "", y_col)[:18]
        if not clean_name:
            clean_name = f"col{i}"

        ent = f"{clean_name}_entity"
        ser = f"{clean_name}_series"

        # Sanitize the string literal for SMT-LIB (keep only printable ASCII, swap double quotes)
        safe_y_col = re.sub(r"[^\x20-\x7E]", "", y_col).replace('"', "'")

        declarations.append(f"(declare-const {ent} Entity)")
        declarations.append(f"(declare-const {ser} Series)")

        anchors.append(f'(assert (= (name_of {ent}) "{safe_y_col}"))')
        anchors.append(f"(assert (attr {ser} {ent}))")

    declarations.extend(
        [
            "(declare-const max_val Real)",
            "(declare-const min_val Real)",
            "(declare-const target_val Real)",
            "(declare-const cond_bool Bool)",
            "(declare-const is_inc_bool Bool)",
            "(declare-const is_dec_bool Bool)",
        ]
    )

    # Dynamically declare rank entities for List/Ranking questions
    for i in range(1, len(y_cols) + 1):
        declarations.append(f"(declare-const rank{i}_entity Entity)")

    # 4. Extract Data Points and ensure explicit floats (e.g., 0.0)
    for row in data:
        if not row or len(row) <= 1:
            continue
        try:
            x_val = float(row[0])
            x_val_str = f"{x_val:.1f}" if x_val.is_integer() else str(x_val)

            for i, y_str in enumerate(row[1:]):
                if i < len(y_cols):
                    y_val = float(y_str)
                    y_val_str = f"{y_val:.1f}" if y_val.is_integer() else str(y_val)

                    clean_name = re.sub(r"[^a-zA-Z0-9]", "", y_cols[i])[:18]
                    if not clean_name:
                        clean_name = f"col{i}"
                    ser = f"{clean_name}_series"

                    anchors.append(f"(assert (= (f {ser} {x_val_str}) {y_val_str}))")
        except ValueError:
            continue  # Skip non-numeric rows

    return "\n".join(declarations), "\n".join(anchors)

In [ ]:
def reflect(q_obj, image, summary, table, max_retries=3, verbose=False, **gen_kwargs):
    question_text = q_obj.get("question") or q_obj.get("questions")
    question_type = q_obj.get("question_type", "")
    answer_type = q_obj.get("answer_type", "")

    # ==========================================
    # PASS 1: Declarations & Anchors
    # ==========================================
    if table:
        declarations, anchors = parse_table_deterministically(table)
    else:
        declarations, anchors = generate_declarations(
            q_obj, image, summary, table, max_retries=3, verbose=False, **gen_kwargs
        )

    full_kb = f"{declarations}\n{anchors}"

    if verbose:
        print("[KNOWLEDGE BASE EXTRACTED]")
        print(full_kb)
        print("-" * 40)

    # ==========================================
    # PHASE 2: Logic
    # ==========================================
    try:
        dynamic_cfg_pass2 = build_dynamic_phase2_cfg(full_kb)
    except Exception as e:
        return None, f"Failed to compile dynamic Phase 2 grammar: {e}"

    second_pass_text = PROMPT_TEMPLATE_PASS2.format(
        question=question_text,
        question_type=question_type,
        answer_type=answer_type,
        summary=summary,
        table=table,
        preamble=PREAMBLE,
        declarations=declarations,
        anchors=anchors,
        example=EXAMPLES_PASS2.get(answer_type, ""),
    )

    prompt_pass2 = Chat(
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": second_pass_text},
                ],
            }
        ]
    )

    for attempt in range(max_retries):
        logic = model(prompt_pass2, dynamic_cfg_pass2, **gen_kwargs)

        final_smt = f""";; --- [PREAMBLE] ---
{PREAMBLE}
;; --- [KNOWLEDGE BASE] ---
{full_kb}
;; --- [PASS 2: Logic & Execution] Attempt {attempt + 1} ---
{logic}
"""

        success, output = validate_smt(final_smt)

        # Validate answer bindings
        if success:
            if answer_type == "Yes/No" and "AnsBool" not in logic:
                success = False
                output = "LOGICAL ERROR: You forgot to assign your final conclusion to AnsBool. You must include an assertion like (= AnsBool ...)"
            elif (
                answer_type in ["Factoid", "Paragraph"]
                and "AnsString" not in logic
                and "AnsBool" not in logic
            ):
                success = False
                output = "LOGICAL ERROR: You forgot to assign your final conclusion to AnsString or AnsBool."

        if verbose:
            print(
                f"[PASS 2 - Attempt {attempt + 1}]\n[Code]\n{final_smt}\n[Output]\n{output}\n"
            )

        if success:
            return final_smt, output

        # Reflect on failure
        reflection_text = (
            f"The SMT solver failed with this output:\n{output}\n\n"
            f"Fix the logic. You MAY NOT declare new entities or series. Use only the provided variables."
        )
        prompt_pass2.add_assistant_message([{"type": "text", "text": logic}])
        prompt_pass2.add_user_message([{"type": "text", "text": reflection_text}])

    return None, "Failed to reach a valid logical formulation in Pass 2 after retries."

In [ ]:
ANSWER_TYPE = "List"

filename = None
q_index = None
for file_path in CASE_DIR.rglob("*.json"):
    if (
        "content.json" in file_path.name
        or "images" not in str(file_path)
        or ".vscode" in str(file_path)
    ):
        continue

    with open(file_path, "r") as f:
        data = json.load(f)

    sub_key = list(data["bbox"].keys())[0]

    if "vqa" not in data:
        continue

    for i, q_obj in enumerate(data["vqa"][sub_key]):
        if q_obj.get("answer_type", "") == ANSWER_TYPE:
            filename = file_path
            q_index = i
            break

    if filename is not None:
        break

In [ ]:
with filename.open("r") as f:
    data = json.load(f)

img = Image.open(filename.with_suffix(".jpg"))
box = data["bbox"][sub_key]

crop = img.crop((box["x"], box["y"], box["x"] + box["width"], box["y"] + box["height"]))
print(f"Subplot: {sub_key}")
for key, value in data["vqa"][sub_key][q_index].items():
    print(f"{key.replace('_', ' ').title()}: {value}")
display(crop)

In [ ]:
if CATEGORY == "dev":
    with SUMMARY_CACHE_PATH.open("r") as fp:
        summary_cache = json.load(fp)
        summary = summary_cache[data["sample_id"]][sub_key]

    with EXTRACTION_CACHE_PATH.open("r") as fp:
        extraction_cache = json.load(fp)
        table = extraction_cache[data["sample_id"]][sub_key]
else:
    summary = data["summarization"][sub_key]
    table = data["data_extraction"][sub_key]

In [ ]:
md = f"""
# Summary
{summary}

# Table
{table}

**Warning**: During training we will employ the actual and not inferred data!
"""

display(Markdown(md))

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # Use torch.float16 if your GPU is older and doesn't support bfloat16
    bnb_4bit_use_double_quant=True,  # Optional: Saves a bit more memory
    bnb_4bit_quant_type="nf4",  # Optional: Recommended for better performance
)

model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto", quantization_config=bnb_config
    ),
    AutoTokenizer.from_pretrained(MODEL_ID),
)

In [ ]:
reflect(
    data["vqa"][sub_key][q_index],
    crop,
    summary,
    table,
    max_retries=3,
    verbose=True,
    do_sample=True,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    min_p=MIN_P,
    top_p=TOP_P,
    top_k=TOP_K,
    # presence_penalty=PRESENCE_PENALTY, # NOT USED!
    repetition_penalty=REPETITION_PENALTY,
)